# Lab CosyVoice3 + Stage 1-2 manifest

**Model:** Fun-CosyVoice3-0.5B-2512  
**Docs:** https://github.com/FunAudioLLM/CosyVoice  
**Ref giọng:** dung audio lab Kokoro (bf_emma / bm_george) — duoc cho lab.

## Thu tu chay (tung cell)

| Buoc | Cell | Viec |
|------|------|------|
| 1 | Cell 1 | Config duong dan |
| 2 | Cell 2 | Clone/pull ielts-script2audio + prepare manifest |
| 3 | Cell 3 | Clone CosyVoice + pip requirements |
| 4 | Cell 4 | Download weight Fun-CosyVoice3-0.5B-2512 |
| 5 | Cell 5 | Tao ref tu wav Kokoro (hoac upload tay) |
| 6 | Cell 6 | Smoke zero_shot 1 cau EN |
| 7 | Cell 7 | Render Part 1 tu manifest (limit 4) |
| 8 | Cell 8 | Tracking DISPLAY/SPOKEN/audio |
| 9 | Cell 9 | (Tuy) instruct "noi nhanh/cang" |

**Khong** cai stack nay ve laptop. **Khong** sua display_text.


## Cell 1 — Config


In [ ]:
# CELL 1
IELTS_REPO = "https://github.com/phamtu2x5/ielts-script2audio.git"
IELTS_DIR = "/content/ielts-script2audio"
COSY_DIR = "/content/CosyVoice"
BRANCH = "main"
print("OK Cell 1")


## Cell 2 — Repo IELTS + prepare manifest


In [ ]:
# CELL 2
import os
from pathlib import Path

if not Path(IELTS_DIR).exists():
    !git clone --branch {BRANCH} {IELTS_REPO} {IELTS_DIR}
else:
    %cd {IELTS_DIR}
    !git pull origin {BRANCH}
%cd {IELTS_DIR}
!pip -q install -e ".[dev]"

Path("outputs").mkdir(exist_ok=True)
!ielts-s2a prepare examples/part1_valid.json --pretty -o outputs/part1_manifest.json

import json
m = json.loads(Path("outputs/part1_manifest.json").read_text())
assert m["validation"]["valid"] is True
print("OK Cell 2 n_requests=", len(m["requests"]))


## Cell 3 — Clone CosyVoice (docs official)


In [ ]:
# CELL 3 — CosyVoice repo
import os
from pathlib import Path

if not Path(COSY_DIR).exists():
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git {COSY_DIR}
else:
    print("CosyVoice dir exists")

%cd {COSY_DIR}
!git submodule update --init --recursive

# Official: requirements.txt (Colab may take several minutes)
!pip -q install -r requirements.txt
# sox if needed
!apt-get -qq update && apt-get -qq install -y sox libsox-dev || true

print("OK Cell 3 CWD=", os.getcwd())


## Cell 4 — Download Fun-CosyVoice3-0.5B-2512


In [ ]:
# CELL 4 — weights (slow first time)
from huggingface_hub import snapshot_download
from pathlib import Path

model_dir = Path(COSY_DIR) / "pretrained_models" / "Fun-CosyVoice3-0.5B"
model_dir.parent.mkdir(parents=True, exist_ok=True)

snapshot_download(
    "FunAudioLLM/Fun-CosyVoice3-0.5B-2512",
    local_dir=str(model_dir),
)
print("OK Cell 4 model_dir=", model_dir)
print("exists", model_dir.exists())


## Cell 5 — Ref tu Kokoro

**Co dung giọng Kokoro lam ref duoc** (lab): CosyVoice zero-shot clone tembre tu file wav mau.

Cach A (khuyen nghi): upload/copy 2 file da render bang Kokoro:
- spk_a: doan bf_emma (vd seg_0001)
- spk_b: doan bm_george (vd seg_0002)

Cach B: neu dang co lab_audio Kokoro tren may/Drive, tro path vao ben duoi.


In [ ]:
# CELL 5 — prepare ref wavs (Kokoro synthetic refs OK for lab)
from pathlib import Path
import shutil
import json

ref_dir = Path(IELTS_DIR) / "voice_bank" / "cosy_refs"
ref_dir.mkdir(parents=True, exist_ok=True)

# Neu da chay notebooks/colab_build_cosyvoice_refs_from_kokoro.ipynb
# thi spk_a_ref.wav / spk_b_ref.wav da 16kHz + refs_manifest.json san sang.
if (ref_dir / 'refs_manifest.json').is_file():
    print('Found refs_manifest.json — prefabricated CosyVoice refs OK')


# --- CHINH PATH NAY neu ban de wav Kokoro o cho khac ---
# Vi du sau lab Kokoro tren cung Colab:
KOKORO_A = Path(IELTS_DIR) / "lab_audio/kokoro_part1/seg_0001__bf_emma.wav"
KOKORO_B = Path(IELTS_DIR) / "lab_audio/kokoro_part1/seg_0002__bm_george.wav"

# Neu ban upload tay vao /content/:
# KOKORO_A = Path("/content/spk_a_ref.wav")
# KOKORO_B = Path("/content/spk_b_ref.wav")

spk_a = ref_dir / "spk_a_ref.wav"
spk_b = ref_dir / "spk_b_ref.wav"

if KOKORO_A.is_file():
    shutil.copy(KOKORO_A, spk_a)
    print("copied A", KOKORO_A)
else:
    print("MISSING A — upload wav va set KOKORO_A:", KOKORO_A)

if KOKORO_B.is_file():
    shutil.copy(KOKORO_B, spk_b)
    print("copied B", KOKORO_B)
else:
    print("MISSING B — upload wav va set KOKORO_B:", KOKORO_B)

# prompt_text phai khop noi dung ref (theo Part1 fixture)
map_path = Path(IELTS_DIR) / "configs/voice_maps/cosyvoice3_from_kokoro_part1.json"
vmap = json.loads(map_path.read_text())
# cap nhat path tuyet doi cho de
vmap["mapping"]["vp_spk_a"]["ref_wav"] = str(spk_a)
vmap["mapping"]["vp_spk_b"]["ref_wav"] = str(spk_b)
vmap["model_dir"] = str(Path(COSY_DIR) / "pretrained_models" / "Fun-CosyVoice3-0.5B")
out_map = ref_dir / "voice_map_runtime.json"
out_map.write_text(json.dumps(vmap, ensure_ascii=False, indent=2) + "\n")
print("OK Cell 5 map=", out_map)
print("A exists", spk_a.is_file(), "B exists", spk_b.is_file())


## Cell 6 — Smoke 1 cau EN (docs CosyVoice3)


In [ ]:
# CELL 6 — smoke zero_shot EN
import sys
from pathlib import Path
import torchaudio

sys.path.insert(0, str(Path(COSY_DIR) / "third_party" / "Matcha-TTS"))
sys.path.insert(0, COSY_DIR)

from cosyvoice.cli.cosyvoice import AutoModel

model_dir = str(Path(COSY_DIR) / "pretrained_models" / "Fun-CosyVoice3-0.5B")
ref = str(Path(IELTS_DIR) / "voice_bank/cosy_refs/spk_a_ref.wav")
assert Path(ref).is_file(), "Chay Cell 5 / co ref A truoc"

cosyvoice = AutoModel(model_dir=model_dir)
tts_text = "Good morning. This is a short CosyVoice lab test for English."
prompt_text = "You are a helpful assistant.<|endofprompt|>Good morning, City Library. How can I help you?"

out_dir = Path(IELTS_DIR) / "lab_audio/cosyvoice3_smoke"
out_dir.mkdir(parents=True, exist_ok=True)

n = 0
for i, j in enumerate(cosyvoice.inference_zero_shot(
    tts_text, prompt_text, ref, stream=False
)):
    p = out_dir / f"smoke_zero_shot_{i}.wav"
    torchaudio.save(str(p), j["tts_speech"], cosyvoice.sample_rate)
    print("wrote", p)
    n += 1
print("OK Cell 6 files=", n, "sr=", cosyvoice.sample_rate)

# keep model in memory for next cells
globals()["cosyvoice"] = cosyvoice


## Cell 7 — Render manifest Part1 (limit 4)


In [ ]:
# CELL 7 — render from Stage-2 manifest
import json
import sys
from pathlib import Path
import torchaudio

sys.path.insert(0, str(Path(COSY_DIR) / "third_party" / "Matcha-TTS"))
sys.path.insert(0, COSY_DIR)

from cosyvoice.cli.cosyvoice import AutoModel

IELTS = Path(IELTS_DIR)
manifest = json.loads((IELTS / "outputs/part1_manifest.json").read_text())
vmap = json.loads((IELTS / "voice_bank/cosy_refs/voice_map_runtime.json").read_text())
model_dir = vmap["model_dir"]

if "cosyvoice" not in globals():
    cosyvoice = AutoModel(model_dir=model_dir)
else:
    cosyvoice = globals()["cosyvoice"]

out_dir = IELTS / "lab_audio/cosyvoice3_part1"
out_dir.mkdir(parents=True, exist_ok=True)

LIMIT = 4  # smoke; set None for all
requests = list(manifest.get("requests") or [])
if LIMIT:
    requests = requests[:LIMIT]

report = {
    "backend": "fun-cosyvoice3-0.5b-2512",
    "mode": "zero_shot",
    "ref_source": "kokoro_lab_wavs",
    "ok_count": 0,
    "fail_count": 0,
    "segments": [],
}

for req in requests:
    seg_id = req["segment_id"]
    vp = req["voice_profile_id"]
    spoken = req.get("spoken_text") or req.get("display_text") or ""
    display = req.get("display_text") or ""
    entry = (vmap.get("mapping") or {}).get(vp) or {}
    ref_wav = entry.get("ref_wav")
    prompt_text = entry.get("prompt_text") or "You are a helpful assistant.<|endofprompt|>Hello."
    item = {
        "segment_id": seg_id,
        "speaker_id": req.get("speaker_id"),
        "voice_profile_id": vp,
        "display_text": display,
        "spoken_text": spoken,
        "ref_wav": ref_wav,
        "status": "PENDING",
        "output_filename": None,
        "error": None,
    }
    try:
        if not ref_wav or not Path(ref_wav).is_file():
            raise FileNotFoundError(f"missing ref for {vp}: {ref_wav}")
        chunks = []
        for j in cosyvoice.inference_zero_shot(spoken, prompt_text, ref_wav, stream=False):
            chunks.append(j["tts_speech"])
        if not chunks:
            raise RuntimeError("no audio")
        import torch
        audio = torch.cat(chunks, dim=1) if len(chunks) > 1 else chunks[0]
        out_name = f"{seg_id}__{vp}.wav"
        out_path = out_dir / out_name
        torchaudio.save(str(out_path), audio, cosyvoice.sample_rate)
        item["status"] = "EXECUTED"
        item["output_filename"] = out_name
        report["ok_count"] += 1
    except Exception as e:
        item["status"] = "FAILED"
        item["error"] = f"{type(e).__name__}: {e}"
        report["fail_count"] += 1
    report["segments"].append(item)
    print(item["segment_id"], item["status"], item.get("error") or item.get("output_filename"))

(out_dir / "lab_render_report.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2) + "\n"
)
print("OK Cell 7 ok=", report["ok_count"], "fail=", report["fail_count"])


## Cell 8 — Tracking


In [ ]:
# CELL 8 — track
from pathlib import Path
from IPython.display import Audio, display
import json

rep = Path(IELTS_DIR) / "lab_audio/cosyvoice3_part1/lab_render_report.json"
assert rep.is_file(), "Chay Cell 7 truoc"
report = json.loads(rep.read_text())
audio_dir = rep.parent
for i, seg in enumerate(report.get("segments") or [], 1):
    print("=" * 72)
    print(f"[{i}] {seg.get('segment_id')} | {seg.get('speaker_id')} | {seg.get('status')}")
    print("DISPLAY:", seg.get("display_text"))
    print("SPOKEN :", seg.get("spoken_text"))
    print("REF    :", seg.get("ref_wav"))
    fn = seg.get("output_filename")
    wav = audio_dir / fn if fn else None
    if wav and wav.is_file():
        display(Audio(filename=str(wav)))
    elif seg.get("error"):
        print("ERROR", seg["error"])
print("OK Cell 8")


## Cell 9 — (Tuy) instruct nhanh/cang tren 1 cau


In [ ]:
# CELL 9 — instruct smoke (emotion/rate)
import sys
from pathlib import Path
import torchaudio

sys.path.insert(0, str(Path(COSY_DIR) / "third_party" / "Matcha-TTS"))
sys.path.insert(0, COSY_DIR)

from cosyvoice.cli.cosyvoice import AutoModel

if "cosyvoice" not in globals():
    cosyvoice = AutoModel(model_dir=str(Path(COSY_DIR) / "pretrained_models/Fun-CosyVoice3-0.5B"))

ref = str(Path(IELTS_DIR) / "voice_bank/cosy_refs/spk_b_ref.wav")
tts_text = "I need this sorted out today. The booking is wrong and I am not happy."
instruct = (
    "You are a helpful assistant. Speak slightly faster, more urgent and tense."
    "<|endofprompt|>"
)
out_dir = Path(IELTS_DIR) / "lab_audio/cosyvoice3_smoke"
out_dir.mkdir(parents=True, exist_ok=True)

for i, j in enumerate(cosyvoice.inference_instruct2(tts_text, instruct, ref, stream=False)):
    p = out_dir / f"smoke_instruct_tense_{i}.wav"
    torchaudio.save(str(p), j["tts_speech"], cosyvoice.sample_rate)
    print("wrote", p)
print("OK Cell 9 — so voi zero_shot: co gấp/cang hon khong?")


Xong lab. not_selected. Khong sua script. Ref Kokoro chi dung cho lab.
